# B2-019-attention-transformers — Practice p22 — Solution

**Type:** scenario · **Difficulty:** intro · **Concepts:** attention-mask

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb).

## Independent solution

Encoder self-attention uses source key validity shaped (B,1,1,Ns), broadcasting across heads and query rows; allowed means the source key is real. Decoder self-attention combines target key validity (B,1,1,Nt) with a causal mask (1,1,Nt,Nt), yielding (B,1,Nt,Nt); an entry is allowed iff key j is real and j<=i. Cross-attention reuses source key validity (B,1,1,Ns) and broadcasts to (B,h,Nt,Ns). Query validity is a separate output/loss policy. Reject any real query row with no allowed key; padded query rows may be zeroed and excluded from loss by an explicit policy. Confusing query and key padding can (1) mask an entire padded query row yet still let real queries read padded key/value columns, or (2) apply a query-valid vector on the key axis, hiding unrelated keys or causing wrong broadcasting.

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 0.0
RTOL = 0.0
source_valid = np.array([[True, True, False], [True, False, False]])
target_valid = np.array([[True, True, True, True], [True, True, True, False]])
causal = np.tril(np.ones((1, 1, 4, 4), dtype=bool))
encoder_allowed = source_valid[:, None, None, :]
decoder_allowed = causal & target_valid[:, None, None, :]
cross_allowed = source_valid[:, None, None, :]
encoder_broadcast = np.broadcast_to(encoder_allowed, (2, 2, 3, 3))
decoder_broadcast = np.broadcast_to(decoder_allowed, (2, 2, 4, 4))
cross_broadcast = np.broadcast_to(cross_allowed, (2, 2, 4, 3))

### Answer check

In [ ]:
assert encoder_allowed.shape == (2, 1, 1, 3)
assert decoder_allowed.shape == (2, 1, 4, 4)
assert cross_allowed.shape == (2, 1, 1, 3)
assert encoder_broadcast.shape == (2, 2, 3, 3)
assert decoder_broadcast.shape == (2, 2, 4, 4)
assert cross_broadcast.shape == (2, 2, 4, 3)
assert np.all(np.any(decoder_broadcast, axis=-1))
assert np.all(np.any(cross_broadcast, axis=-1))
assert not target_valid[1, 3]
assert not decoder_broadcast[1, :, :, 3].any()
assert not decoder_broadcast[:, :, :, 3][1].any()